| version | description                                            | LB    |
|---------|-------------------------------------------------------|-------|
| 23      | v4 + pLDDT sort                                        | 0.436 |
| 32      | v23 + GUDHI H1 PH Rerank + Noise                     | 0.447 |
| 33      | v32 No noise (counterproductive)                                | 0.418 |
| 34      | v23 + Noise only (counterproductive)                              | 0.426 |
| 35      | v32 + Official gudhi installation                            | TBC   |
| 36      | v35 + Auto-draw H1 persistent diagram                     | TBC   |


# GUDHI H1 Persistent Homology Reranking

## Overview

Applies Persistent Homology (PH) to the C1' coordinate arrays of Protenix samples
and reranks them so that the topologically most representative structure is placed first.
TBM sample ordering is never modified.

---

## Pipeline
```
Phase 1 : TBM          → up to N_SAMPLE predictions (by similarity)
Phase 2 : Protenix     → fills remaining slots (sorted by pLDDT descending)
          ↓
          PH rerank    → applied only within the Protenix block
Phase 3 : de-novo      → last-resort fallback
```

---

## Formulation

### Step 1 — H1 Feature Extraction

For candidate $C_i \in \mathbb{R}^{L \times 3}$, compute the pairwise distance matrix $D_{jk} = \|C_i^{(j)} - C_i^{(k)}\|_2$, build a Rips filtration, and extract H1 birth–death pairs:

$$\text{pairs} = \{(b_k,\, d_k) \mid d_k - b_k \geq \epsilon_{\min}\}$$

After scale normalization, vectorize into a feature vector:

$$\phi(C_i) = \bigl[\underbrace{p_1,\ldots,p_{16}}_{\text{top-16 persistence}},\ \underbrace{\text{8 statistics}}_{\text{mean, std, etc.}},\ \underbrace{h_b}_{\text{birth histogram}},\ \underbrace{h_p}_{\text{persistence histogram}}\bigr] \in \mathbb{R}^{64}$$

### Step 2 — Reference Feature (Median Ensemble)

$$\phi_{\text{ref}} = \operatorname{median}\bigl(\phi(C_0),\ldots,\phi(C_{n-1})\bigr)$$

### Step 3 — Scoring

$$s_i^{\text{PH}} = -\|\phi(C_i) - \phi_{\text{ref}}\|_2 - \beta\, B(C_i) - \gamma\, K(C_i)$$

$$B(C_i) = \frac{1}{L-1}\sum_{j=1}^{L-1}\bigl(\|C_i^{(j+1)}-C_i^{(j)}\|_2 - 6.0\bigr)^2 \quad\text{(bond length penalty,}\ \beta=0.25\text{)}$$

$$K(C_i) = \frac{1}{|M|}\sum_{(j,k)\in M}(2.2 - d_{jk})^2 \quad M=\{d_{jk}<2.2,\ |j-k|>2\} \quad\text{(clash penalty,}\ \gamma=0.20\text{)}$$

### Step 4 — Final Score and Reranking

Normalize the PH scores and blend with the original rank order:

$$\tilde{s}_i^{\text{PH}} = \frac{s_i^{\text{PH}} - \mu}{\sigma}$$

$$\text{score}_i = \underbrace{-0.035 \cdot i}_{\text{original rank penalty}} + \underbrace{0.010 \cdot \tilde{s}_i^{\text{PH}}}_{\text{PH signal}}$$

Candidates are sorted by $\text{score}_i$ in descending order.

---

## Intuition

> **"Place the candidate whose topology is closest to the median of all candidates at rank 1."**

The PH signal weight $0.010$ is roughly 28% of the rank penalty $0.035$,
so reranking is a mild correction that rarely causes large rank swaps.

---

## Parameters

| Parameter | Value | Description |
|---|---|---|
| `PH_BLEND_LAMBDA` | 0.010 | Blend weight for PH signal |
| `PH_BLEND_BASE_GAP` | 0.035 | Per-rank penalty coefficient |
| `PH_ALPHA` | 1.0 | PH score weight |
| `PH_BETA` | 0.25 | Bond length penalty weight |
| `PH_GAMMA` | 0.20 | Clash penalty weight |
| `PH_MAX_POINTS` | 96 | Subsampling cap |
| `PH_H1_TOPK` | 16 | Number of top persistence values |
| `PH_BETTI_BINS` | 24 | Histogram bin count |
| `PH_MAX_EDGE_MULT` | 3.5 | Max edge length multiplier for Rips |
| `PH_MIN_PERSISTENCE` | 1e-3 | Minimum persistence threshold |
| `EXPECTED_C1_STEP` | 6.0 Å | Ideal C1'–C1' distance |
| `CLASH_DIST` | 2.2 Å | Clash detection distance |

In [ ]:
# =============
# 1) Install
# =============
# Assumes Kaggle offline wheels (adjust the path to match your wheel directory)
!pip install --no-index --find-links=/kaggle/input/datasets/takahiro3110/gungun-cp312/gudhi_wheels_cp312/wheels gudhi

Reference

https://www.kaggle.com/code/llkh0a/stanford-rna-3d-folding-part-2-protenix-tbm

In [ ]:
!pip install --no-index --no-deps /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl

!pip install --no-index --no-deps /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl

!pip install --no-index --no-deps /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl

!pip install --no-index --find-links=/kaggle/input/datasets/takahiro3110/gungun-cp312/gudhi_wheels_cp312/wheels gudhi


In [ ]:
import os
import sys
import pandas as pd

# ── Local vs Kaggle mode ─────────────────────────────────────────────────────
# On Kaggle competition rerun, KAGGLE_IS_COMPETITION_RERUN is set to a truthy value.
# When running locally we do NOT exit — instead we cap the test set to a small
# number of samples so the notebook finishes quickly.

IS_KAGGLE = True #bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN", ""))

# How many test samples to use when running locally
LOCAL_N_SAMPLES = None

if IS_KAGGLE:
    print("Running in KAGGLE COMPETITION mode — all test targets will be processed.")
else:
    print(f"Running in LOCAL mode — only the first {LOCAL_N_SAMPLES} test targets "
          f"will be processed to save time.")

In [ ]:
import gc
import json
import os
import time

os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio.Align import PairwiseAligner
from tqdm import tqdm

In [ ]:
def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]
    
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
        
    # Heuristic fallback: check which index gives us roughly N_token atoms
    n_tokens = data.get("N_token", torch.tensor(0)).item()
    mask11 = (f["atom_to_tokatom_idx"] == 11).bool()
    mask12 = (f["atom_to_tokatom_idx"] == 12).bool()
    
    c11 = mask11.sum().item()
    c12 = mask12.sum().item()
    
    # Return the one closer to N_tokens (likely one per residue)
    if abs(c11 - n_tokens) < abs(c12 - n_tokens):
        return mask11
    else:
        return mask12

In [ ]:

# ─────────────── Paths & Constants ───────────────────────────────────────────
DATA_BASE              = "/kaggle/input/stanford-rna-3d-folding-2"
DEFAULT_TEST_CSV       = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV      = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_TRAIN_LBLS     = f"{DATA_BASE}/train_labels.csv"
DEFAULT_VAL_CSV        = f"{DATA_BASE}/validation_sequences.csv"
DEFAULT_VAL_LBLS       = f"{DATA_BASE}/validation_labels.csv"
DEFAULT_OUTPUT         = "/kaggle/working/submission.csv"

DEFAULT_CODE_DIR = (
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted"
    "/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
)
DEFAULT_ROOT_DIR = DEFAULT_CODE_DIR

MODEL_NAME    = "protenix_base_20250630_v1.0.0"
N_SAMPLE      = 5
SEED          = 42
MAX_SEQ_LEN   = int(os.environ.get("MAX_SEQ_LEN",   "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP",  "128"))

# TBM quality thresholds — sequences below these get routed to Protenix
MIN_SIMILARITY       = float(os.environ.get("MIN_SIMILARITY",       "0.0"))
MIN_PERCENT_IDENTITY = float(os.environ.get("MIN_PERCENT_IDENTITY", "50.0"))

# Set False to skip Protenix and use de-novo fallback instead
USE_PROTENIX = True


def parse_bool(value: str, default: bool = False) -> str:
    v = str(value).strip().lower()
    if v in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if v in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"


USE_MSA      = parse_bool(os.environ.get("USE_MSA",      "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA  = parse_bool(os.environ.get("USE_RNA_MSA",  "true"))

MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))


# ─────────────── General Utilities ───────────────────────────────────────────
def seed_everything(seed: int) -> None:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


def resolve_paths():
    test_csv   = os.environ.get("TEST_CSV",           DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV",     DEFAULT_OUTPUT)
    code_dir   = os.environ.get("PROTENIX_CODE_DIR",  DEFAULT_CODE_DIR)
    root_dir   = os.environ.get("PROTENIX_ROOT_DIR",  DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


def ensure_required_files(root_dir: str) -> None:
    for p, name in [
        (Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt",          "checkpoint"),
        (Path(root_dir) / "common" / "components.cif",                "CCD file"),
        (Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl",  "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")


# ─────────────── Protenix Input / Config Helpers ─────────────────────────────
def build_input_json(df: pd.DataFrame, json_path: str) -> None:
    data = [
        {
            "name": row["target_id"],
            "covalent_bonds": [],
            "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}],
        }
        for _, row in df.iterrows()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict):
                deep_update(t[k], v)
            else:
                t[k] = v

    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]
    
    # CASE A: center_atom_mask exists
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
        
    # CASE B: Use atom_name
    if "atom_name" in f:
        # Check against "C1'" (byte encoded or string?)
        # For now assume typical behavior is center_atom_mask is present.
        pass

    # CASE C: atom_to_tokatom_idx fallback
    # The index for C1' is typically 11 or 12 depending on featurizer.
    # Let's try to match exactly C1' if possible.
    # But usually 'centre_atom_mask' should be there.
    
    # If we fall through, assume standard mask
    return (f["atom_to_tokatom_idx"] == 11).bool()


def get_feature_c1_mask(data: dict) -> torch.Tensor:
    f = data["input_feature_dict"]
    if "centre_atom_mask" in f:
        return f["centre_atom_mask"].long() == 1
    return f["atom_to_tokatom_idx"].long() == 12


def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    """coords shape: (N_SAMPLE, seq_len, 3)"""
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows


def pad_samples(coords: np.ndarray, n: int) -> np.ndarray:
    if coords.shape[0] >= n:
        return coords[:n]
    if coords.shape[0] == 0:
        return np.zeros((n, coords.shape[1], 3), dtype=coords.dtype)
    extra = np.repeat(coords[:1], n - coords.shape[0], axis=0)
    return np.concatenate([coords, extra], axis=0)


def split_into_chunks(seq_len: int, max_len: int, overlap: int) -> list:
    """Split a sequence into overlapping (start, end) chunks."""
    if seq_len <= max_len:
        return [(0, seq_len)]
    chunks = []
    step = max_len - overlap
    pos = 0
    while pos < seq_len:
        end = min(pos + max_len, seq_len)
        chunks.append((pos, end))
        if end == seq_len:
            break
        pos += step
    return chunks


def kabsch_align(P: np.ndarray, Q: np.ndarray):
    """Compute optimal rotation R and translation t so that  R @ P + t ≈ Q."""
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    Pc = P - centroid_P
    Qc = Q - centroid_Q
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    S = np.eye(3)
    if d < 0:
        S[2, 2] = -1
    R = Vt.T @ S @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t


def stitch_chunk_coords(chunk_coords_list: list,
                        chunk_ranges: list,
                        seq_len: int) -> np.ndarray:
    """
    Merge overlapping chunk coordinates into a full sequence geometry.
    Applies Kabsch alignment on overlapping residues, and smoothly
    blends the coordinates using a linear weight ramp.
    """
    if len(chunk_coords_list) == 1:
        coords = chunk_coords_list[0]
        if coords.shape[0] >= seq_len:
            return coords[:seq_len]
        out = np.zeros((seq_len, 3), dtype=coords.dtype)
        out[:coords.shape[0]] = coords
        return out

    # Start with the first chunk aligned to itself (identity)
    aligned = [chunk_coords_list[0].copy()]

    for i in range(1, len(chunk_coords_list)):
        prev_start, prev_end = chunk_ranges[i - 1]
        cur_start, cur_end = chunk_ranges[i]

        ov_start = cur_start
        ov_end = min(prev_end, cur_end)
        ov_len = ov_end - ov_start

        if ov_len < 3:
            # Cannot align reliably, just trust the coordinates as-is
            aligned.append(chunk_coords_list[i].copy())
            continue

        prev_ov = aligned[i - 1][ov_start - prev_start: ov_end - prev_start]
        cur_ov = chunk_coords_list[i][ov_start - cur_start: ov_end - cur_start]

        # Ignore invalid residues (e.g. padding/blank)
        valid = ~(np.isnan(prev_ov).any(axis=1) | np.isnan(cur_ov).any(axis=1))
        if valid.sum() < 3:
            aligned.append(chunk_coords_list[i].copy())
            continue

        # Align current chunk to previous chunk using only the overlap region
        R, t = kabsch_align(cur_ov[valid], prev_ov[valid])
        transformed = (chunk_coords_list[i] @ R.T) + t
        aligned.append(transformed)

    # Blend them together
    full = np.zeros((seq_len, 3), dtype=np.float64)
    weights = np.zeros(seq_len, dtype=np.float64)

    for i, ((s, e), coords) in enumerate(zip(chunk_ranges, aligned)):
        chunk_len = coords.shape[0]
        actual_end = min(s + chunk_len, seq_len)
        used_len = actual_end - s

        w = np.ones(used_len, dtype=np.float64)

        if i > 0:
            ov_start = s
            ov_end = min(chunk_ranges[i - 1][1], e)
            ramp_len = ov_end - ov_start
            if ramp_len > 0:
                w[:ramp_len] = np.linspace(0.0, 1.0, ramp_len)

        if i < len(chunk_ranges) - 1:
            next_s = chunk_ranges[i + 1][0]
            ramp_start = next_s - s
            ramp_len = actual_end - next_s
            if ramp_len > 0 and ramp_start < used_len:
                w[ramp_start:used_len] = np.linspace(1.0, 0.0, ramp_len)

        full[s:actual_end] += coords[:used_len] * w[:, None]
        weights[s:actual_end] += w

    mask = weights > 0
    full[mask] /= weights[mask, None]

    return full


# ─────────────── TBM Core Functions ──────────────────────────────────────────
def _make_aligner() -> PairwiseAligner:
    al = PairwiseAligner()
    al.mode                           = "global"
    al.match_score                    = 2
    al.mismatch_score                 = -1.5
    al.open_gap_score                 = -8
    al.extend_gap_score               = -0.4
    al.query_left_open_gap_score      = -8
    al.query_left_extend_gap_score    = -0.4
    al.query_right_open_gap_score     = -8
    al.query_right_extend_gap_score   = -0.4
    al.target_left_open_gap_score     = -8
    al.target_left_extend_gap_score   = -0.4
    al.target_right_open_gap_score    = -8
    al.target_right_extend_gap_score  = -0.4
    return al


_aligner = _make_aligner()


def parse_stoichiometry(stoich: str) -> list:
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";")
            for ch, cnt in [part.split(":")]]


def parse_fasta(fasta_content: str) -> dict:
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(parts)
            cur = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(parts)
    return out


def get_chain_segments(row) -> list:
    seq    = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_sq = row.get("all_sequences", "")
    if (pd.isna(stoich) or pd.isna(all_sq)
            or str(stoich).strip() == "" or str(all_sq).strip() == ""):
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_sq)
        order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base)))
                pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]


def build_segments_map(df: pd.DataFrame) -> tuple:
    seg_map, stoich_map = {}, {}
    for _, r in df.iterrows():
        tid               = r["target_id"]
        seg_map[tid]      = get_chain_segments(r)
        raw_s             = r.get("stoichiometry", "")
        stoich_map[tid]   = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map


def process_labels(labels_df: pd.DataFrame) -> dict:
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords


def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs, t_segs = alignment.aligned
    aq, at, qi, ti = [], [], 0, 0
    for (qs, qe), (ts, te) in zip(q_segs, t_segs):
        while qi < qs: aq.append(query_seq[qi]);    at.append("-");              qi += 1
        while ti < ts: aq.append("-");              at.append(template_seq[ti]); ti += 1
        for qp, tp in zip(range(qs, qe), range(ts, te)):
            aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi, ti = qe, te
    while qi < len(query_seq):    aq.append(query_seq[qi]);    at.append("-");              qi += 1
    while ti < len(template_seq): aq.append("-");              at.append(template_seq[ti]); ti += 1
    return "".join(aq), "".join(at)


def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    results = []
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict:
            continue
        if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > 0.3:
            continue
        aln       = next(iter(_aligner.align(query_seq, tseq)))
        norm_s    = aln.score / (2 * min(len(query_seq), len(tseq)))
        identical = sum(
            1 for (qs, qe), (ts, te) in zip(*aln.aligned)
            for qp, tp in zip(range(qs, qe), range(ts, te))
            if query_seq[qp] == tseq[tp]
        )
        pct_id = 100 * identical / len(query_seq)
        aq, at = _build_aligned_strings(query_seq, tseq, aln)
        results.append((tid, tseq, norm_s, train_coords_dict[tid], pct_id, aq, at))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]


def adapt_template_to_query(query_seq, template_seq, template_coords) -> np.ndarray:
    aln        = next(iter(_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan)
    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs):
            new_coords[qs:qe] = chunk
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            nv = next((j for j in range(i + 1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv)
                new_coords[i] = (1 - w) * new_coords[pv] + w * new_coords[nv]
            elif pv >= 0:
                new_coords[i] = new_coords[pv] + [3, 0, 0]
            elif nv >= 0:
                new_coords[i] = new_coords[nv] + [3, 0, 0]
            else:
                new_coords[i] = [i * 3, 0, 0]
    return np.nan_to_num(new_coords)


def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=2) -> np.ndarray:
    X        = coords.copy()
    segments = segments_map.get(target_id, [(0, len(X))])
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)
    for _ in range(passes):
        for s, e in segments:
            C = X[s:e]; L = e - s
            if L < 3:
                continue
            # bond i–i+1  ~5.95 Å
            d    = C[1:] - C[:-1]; dist = np.linalg.norm(d, axis=1) + 1e-6
            adj  = d * ((5.95 - dist) / dist)[:, None] * (0.22 * strength)
            C[:-1] -= adj; C[1:] += adj
            # soft i–i+2  ~10.2 Å
            d2   = C[2:] - C[:-2]; d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((10.2 - d2n) / d2n)[:, None] * (0.10 * strength)
            C[:-2] -= adj2; C[2:] += adj2
            # Laplacian smoothing
            C[1:-1] += (0.06 * strength) * (0.5 * (C[:-2] + C[2:]) - C[1:-1])
            # self-avoidance
            if L >= 25:
                idx  = np.linspace(0, L - 1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P    = C[idx]; diff = P[:, None, :] - P[None, :, :]
                dm   = np.linalg.norm(diff, axis=2) + 1e-6
                sep  = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    C[idx] += (0.015 * strength) * vec
            X[s:e] = C
    return X


def _rotmat(axis, ang):
    a = np.asarray(axis, float); a /= np.linalg.norm(a) + 1e-12
    x, y, z = a; c, s = np.cos(ang), np.sin(ang); CC = 1 - c
    return np.array([[c+x*x*CC, x*y*CC-z*s, x*z*CC+y*s],
                     [y*x*CC+z*s, c+y*y*CC, y*z*CC-x*s],
                     [z*x*CC-y*s, z*y*CC+x*s, c+z*z*CC]])


def apply_hinge(coords, seg, rng, deg=22):
    s, e = seg; L = e - s
    if L < 30: return coords
    pivot = s + int(rng.integers(10, L - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
    X = coords.copy(); p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X


def jitter_chains(coords, segs, rng, deg=12, trans=1.5):
    X = coords.copy(); gc_ = X.mean(0, keepdims=True)
    for s, e in segs:
        R     = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
        shift = rng.normal(size=3); shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, trans))
        c     = X[s:e].mean(0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(0, keepdims=True) - gc_
    return X


def smooth_wiggle(coords, segs, rng, amp=0.8):
    X = coords.copy()
    for s, e in segs:
        L = e - s
        if L < 20: continue
        ctrl = np.linspace(0, L - 1, 6); disp = rng.normal(0, amp, (6, 3)); t = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return X


def generate_rna_structure(sequence: str, seed=None) -> np.ndarray:
    """Idealized A-form RNA helix — last-resort de-novo fallback."""
    if seed is not None:
        np.random.seed(seed)
    n = len(sequence); coords = np.zeros((n, 3))
    for i in range(n):
        ang = i * 0.6
        coords[i] = [10.0 * np.cos(ang), 10.0 * np.sin(ang), i * 2.5]
    return coords


# ─────────────── TBM Phase ───────────────────────────────────────────────────
def tbm_phase(test_df, train_seqs_df, train_coords_dict, segments_map):
    """
    Phase 1 — Template-Based Modeling.

    Returns
    -------
    template_predictions : {target_id: [np.ndarray(seq_len, 3), ...]}
        0 to N_SAMPLE predictions per target, from real templates.
    protenix_queue : {target_id: (n_needed, full_sequence)}
        Targets that still need more predictions.
    """
    print(f"\n{'='*60}")
    print(f"PHASE 1: Template-Based Modeling")
    print(f"  MIN_SIMILARITY = {MIN_SIMILARITY}  |  MIN_PCT_IDENTITY = {MIN_PERCENT_IDENTITY}")
    print(f"{'='*60}")
    t0 = time.time()

    template_predictions: dict = {}
    protenix_queue:       dict = {}

    for _, row in test_df.iterrows():
        tid = row["target_id"]
        seq = row["sequence"]
        segs = segments_map.get(tid, [(0, len(seq))])

        similar = find_similar_sequences_detailed(seq, train_seqs_df, train_coords_dict, top_n=30)
        preds   = []
        used    = set()

        for i, (tmpl_id, tmpl_seq, sim, tmpl_coords, pct_id, _, _) in enumerate(similar):
            if len(preds) >= N_SAMPLE:
                break
            if sim < MIN_SIMILARITY or pct_id < MIN_PERCENT_IDENTITY:
                break           # list is sorted by sim, so no point continuing
            if tmpl_id in used:
                continue

            rng     = np.random.default_rng((row.name * 10000000000 + i * 10007) % (2**32))
            adapted = adapt_template_to_query(seq, tmpl_seq, tmpl_coords)

            # Diversity transforms (same strategy as the 0-409 TBM notebook)
            slot = len(preds)
            if slot == 0:
                X = adapted
            elif slot == 1:
                X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
            elif slot == 2:
                longest = max(segs, key=lambda se: se[1] - se[0])
                X = apply_hinge(adapted, longest, rng)
            elif slot == 3:
                X = jitter_chains(adapted, segs, rng)
            else:
                X = smooth_wiggle(adapted, segs, rng)

            refined = adaptive_rna_constraints(X, tid, segments_map, confidence=sim)
            preds.append(refined)
            used.add(tmpl_id)

        template_predictions[tid] = preds
        n_needed = N_SAMPLE - len(preds)
        if n_needed > 0:
            protenix_queue[tid] = (n_needed, seq)
            print(f"  {tid} ({len(seq)} nt): {len(preds)} TBM → need {n_needed} from Protenix")
        else:
            print(f"  {tid} ({len(seq)} nt): all {N_SAMPLE} from TBM ✓")

    elapsed = time.time() - t0
    n_full  = len(test_df) - len(protenix_queue)
    print(f"\nPhase 1 done in {elapsed:.1f}s")
    print(f"  Fully covered by TBM : {n_full}")
    print(f"  Need Protenix        : {len(protenix_queue)}")
    return template_predictions, protenix_queue



# ============================================================
# GUDHI-based H1 PH rerank for RNA 3D candidates
# ============================================================
# Notes:
# - If gudhi is unavailable, reranking is skipped safely.
# - To enable online install in environments that allow it, uncomment:
#     !pip -q install gudhi
# ============================================================

PH_RERANK_ENABLED = True
PH_MODE = "gudhi_h1"

# PH is used only as a small auxiliary signal on top of the original candidate order.
# final_score = base_order_score + PH_BLEND_LAMBDA * normalized_ph_score
PH_BLEND_LAMBDA   = 0.080
PH_BLEND_BASE_GAP = 0.035

PH_ALPHA = 1.0
PH_BETA  = 0.25   # bond penalty weight
PH_GAMMA = 0.20   # clash penalty weight

PH_MAX_POINTS = 96
PH_H1_TOPK = 16
PH_BETTI_BINS = 24
PH_MAX_EDGE_MULT = 3.5
PH_MIN_PERSISTENCE = 1e-3
PH_VERBOSE = False

EXPECTED_C1_STEP = 6.0
CLASH_DIST = 2.2

try:
    import gudhi as gd
    GUDHI_AVAILABLE = True
except Exception as _gudhi_exc:
    gd = None
    GUDHI_AVAILABLE = False
    print(f"[WARN] gudhi is not available. PH rerank will be skipped. ({_gudhi_exc})")

def _safe_valid_coords(coords: np.ndarray) -> np.ndarray:
    x = np.asarray(coords, dtype=np.float32)
    if x.ndim != 2 or x.shape[1] != 3:
        return np.zeros((0, 3), dtype=np.float32)
    mask = np.isfinite(x).all(axis=1)
    return x[mask]

def _pairwise_distances(x: np.ndarray) -> np.ndarray:
    diff = x[:, None, :] - x[None, :, :]
    D = np.sqrt(np.sum(diff * diff, axis=-1))
    return D.astype(np.float32)

def _nearest_neighbor_scale_from_D(D: np.ndarray) -> float:
    n = D.shape[0]
    if n <= 1:
        return 1.0
    A = D.copy()
    np.fill_diagonal(A, np.inf)
    nn = np.min(A, axis=1)
    s = float(np.median(nn[np.isfinite(nn)])) if np.isfinite(nn).any() else 1.0
    return max(s, 1e-6)

def _subsample_coords(x: np.ndarray, max_points: int = PH_MAX_POINTS) -> np.ndarray:
    n = len(x)
    if n <= max_points:
        return x
    idx = np.linspace(0, n - 1, max_points).round().astype(int)
    idx = np.unique(idx)
    return x[idx]

def bond_length_penalty(coords: np.ndarray, expected_step: float = EXPECTED_C1_STEP) -> float:
    x = _safe_valid_coords(coords)
    if len(x) <= 1:
        return 0.0
    d = np.sqrt(np.sum((x[1:] - x[:-1]) ** 2, axis=1))
    return float(np.mean((d - expected_step) ** 2))

def clash_penalty(coords: np.ndarray, clash_dist: float = CLASH_DIST) -> float:
    x = _safe_valid_coords(coords)
    n = len(x)
    if n <= 3:
        return 0.0
    D = _pairwise_distances(x)
    pen = 0.0
    cnt = 0
    for i in range(n):
        for j in range(i + 2, n):
            dij = float(D[i, j])
            if dij < clash_dist:
                pen += (clash_dist - dij) ** 2
                cnt += 1
    return pen / cnt if cnt > 0 else 0.0

def _extract_h1_pairs_from_simplex_tree(st, min_persistence: float = PH_MIN_PERSISTENCE):
    pairs = []
    pers = st.persistence()
    for dim, bd in pers:
        if dim != 1:
            continue
        b, d = float(bd[0]), float(bd[1])
        if not np.isfinite(d):
            continue
        p = d - b
        if p < min_persistence:
            continue
        pairs.append((b, d, p))
    return pairs

def _h1_feature_from_pairs(
    h1_pairs: list[tuple[float, float, float]],
    topk: int = PH_H1_TOPK,
    bins: int = PH_BETTI_BINS,
    scale: float = 1.0,
) -> np.ndarray:
    if len(h1_pairs) == 0:
        return np.zeros(topk + 8 + bins + bins, dtype=np.float32)

    arr = np.asarray(h1_pairs, dtype=np.float32)
    births = arr[:, 0] / max(scale, 1e-6)
    deaths = arr[:, 1] / max(scale, 1e-6)
    pers   = arr[:, 2] / max(scale, 1e-6)

    pers_sorted = np.sort(pers)[::-1]
    top = np.zeros(topk, dtype=np.float32)
    m = min(topk, len(pers_sorted))
    top[:m] = pers_sorted[:m]

    stats = np.array([
        float(len(pers)),
        float(np.sum(pers)),
        float(np.mean(pers)),
        float(np.std(pers)),
        float(np.max(pers)),
        float(np.mean(births)),
        float(np.std(births)),
        float(np.mean(deaths)),
    ], dtype=np.float32)

    max_birth = max(float(np.max(births)), 1e-6)
    max_pers  = max(float(np.max(pers)), 1e-6)

    birth_hist, _ = np.histogram(
        births, bins=bins, range=(0.0, max_birth), density=False
    )
    pers_hist, _ = np.histogram(
        pers, bins=bins, range=(0.0, max_pers), density=False
    )

    birth_hist = birth_hist.astype(np.float32)
    pers_hist  = pers_hist.astype(np.float32)

    if birth_hist.sum() > 0:
        birth_hist /= birth_hist.sum()
    if pers_hist.sum() > 0:
        pers_hist /= pers_hist.sum()

    return np.concatenate([top, stats, birth_hist, pers_hist], axis=0).astype(np.float32)

def gudhi_h1_feature(coords: np.ndarray) -> np.ndarray:
    if not GUDHI_AVAILABLE:
        raise RuntimeError("gudhi is not installed")

    x = _safe_valid_coords(coords)
    if len(x) < 4:
        return np.zeros(PH_H1_TOPK + 8 + PH_BETTI_BINS + PH_BETTI_BINS, dtype=np.float32)

    x = _subsample_coords(x, PH_MAX_POINTS)
    if len(x) < 4:
        return np.zeros(PH_H1_TOPK + 8 + PH_BETTI_BINS + PH_BETTI_BINS, dtype=np.float32)

    D = _pairwise_distances(x)
    scale = _nearest_neighbor_scale_from_D(D)
    max_edge = max(PH_MAX_EDGE_MULT * scale, 1e-3)

    rips = gd.RipsComplex(distance_matrix=D, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=2)
    h1_pairs = _extract_h1_pairs_from_simplex_tree(st, min_persistence=PH_MIN_PERSISTENCE)

    return _h1_feature_from_pairs(
        h1_pairs,
        topk=PH_H1_TOPK,
        bins=PH_BETTI_BINS,
        scale=scale,
    )

def _make_reference_ph_feature(candidates: list[np.ndarray]) -> np.ndarray:
    feats = []
    for c in candidates:
        try:
            feats.append(gudhi_h1_feature(c))
        except Exception as e:
            if PH_VERBOSE:
                print("[PH ref feature error]", e)

    if len(feats) == 0:
        return np.zeros(PH_H1_TOPK + 8 + PH_BETTI_BINS + PH_BETTI_BINS, dtype=np.float32)

    F = np.stack(feats, axis=0)
    return np.median(F, axis=0).astype(np.float32)

def ph_score_candidate(coords: np.ndarray, ref_feat: np.ndarray) -> tuple[float, dict]:
    feat = gudhi_h1_feature(coords)
    ph_dist = float(np.linalg.norm(feat - ref_feat))
    p_bond  = bond_length_penalty(coords)
    p_clash = clash_penalty(coords)

    score = (
        PH_ALPHA * (-ph_dist)
        - PH_BETA  * p_bond
        - PH_GAMMA * p_clash
    )
    aux = {
        "ph_dist": ph_dist,
        "bond_pen": p_bond,
        "clash_pen": p_clash,
        "score": score,
    }
    return score, aux


def plot_persistence_diagram(h1_pairs: list, target_id: str, sample_idx: int,
                              ref_feat: np.ndarray = None) -> None:
    """
    H1パーシステント図（birth-death plot）を描画する。
    h1_pairs: [(birth, death, persistence), ...]
    """
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import matplotlib.gridspec as gridspec

        fig = plt.figure(figsize=(12, 5))
        fig.suptitle(f"{target_id}  sample_{sample_idx}  —  H1 Persistence Diagram",
                     fontsize=12, fontweight="bold")
        gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

        # ── Left: birth-death plot ────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0])
        if len(h1_pairs) > 0:
            arr    = np.array(h1_pairs)
            births = arr[:, 0]
            deaths = arr[:, 1]
            pers   = arr[:, 2]

            # 寿命でサイズを変える（長寿命ほど大きい点）
            sizes  = 20 + 200 * (pers / (pers.max() + 1e-9))
            sc = ax1.scatter(births, deaths, c=pers, s=sizes,
                             cmap="plasma", alpha=0.8, edgecolors="k", linewidths=0.4)
            plt.colorbar(sc, ax=ax1, label="Persistence")

            # 対角線（birth = death）
            lim = max(deaths.max(), births.max()) * 1.05
            ax1.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.5, label="birth=death")
            ax1.set_xlim(-0.02 * lim, lim)
            ax1.set_ylim(-0.02 * lim, lim)
        else:
            ax1.text(0.5, 0.5, "No H1 pairs", ha="center", va="center",
                     transform=ax1.transAxes, fontsize=11, color="gray")

        ax1.set_xlabel("Birth (Å)")
        ax1.set_ylabel("Death (Å)")
        ax1.set_title("H1  Birth–Death Plot")
        ax1.set_aspect("equal")
        ax1.grid(True, alpha=0.3)

        # ── Right: persistence barcode (寿命順) ──────────────────────────
        ax2 = fig.add_subplot(gs[1])
        if len(h1_pairs) > 0:
            arr_s  = sorted(h1_pairs, key=lambda x: x[2], reverse=True)
            colors = plt.cm.plasma(
                np.linspace(0.9, 0.2, len(arr_s))
            )
            for k, (b, d, p) in enumerate(arr_s):
                ax2.plot([b, d], [k, k], lw=2.5, color=colors[k], alpha=0.85)
            ax2.set_yticks(range(len(arr_s)))
            ax2.set_yticklabels([f"p={p:.2f}" for _, _, p in arr_s], fontsize=7)
            ax2.set_xlabel("Filtration value (Å)")
        else:
            ax2.text(0.5, 0.5, "No H1 pairs", ha="center", va="center",
                     transform=ax2.transAxes, fontsize=11, color="gray")

        ax2.set_title("H1  Barcode (寿命順)")
        ax2.grid(True, alpha=0.3, axis="x")

        plt.tight_layout()
        save_path = f"/kaggle/working/ph_diagram_{target_id}_s{sample_idx}.png"
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"  [PH plot] saved → {save_path}")
    except Exception as e:
        print(f"  [PH plot] failed: {e}")


def _get_h1_pairs_for_coords(coords: np.ndarray) -> list:
    """coords (L,3) から H1 birth-death ペアを返す（可視化用）"""
    if not GUDHI_AVAILABLE:
        return []
    x = _safe_valid_coords(coords)
    if len(x) < 4:
        return []
    x   = _subsample_coords(x, PH_MAX_POINTS)
    D   = _pairwise_distances(x)
    sc  = _nearest_neighbor_scale_from_D(D)
    me  = max(PH_MAX_EDGE_MULT * sc, 1e-3)
    rips = gd.RipsComplex(distance_matrix=D, max_edge_length=me)
    st   = rips.create_simplex_tree(max_dimension=2)
    return _extract_h1_pairs_from_simplex_tree(st, min_persistence=PH_MIN_PERSISTENCE)

def ph_rerank_candidates(
    candidates: list[np.ndarray],
    target_id: str | None = None,
    verbose: bool = False,
) -> tuple[list[np.ndarray], list[dict]]:
    """
    Original candidate order remains the primary signal.
    This function is intended to rerank only a local candidate block
    (now used for the Protenix block only).

    final_score_i = base_score_i + PH_BLEND_LAMBDA * ph_norm_i
    where
        base_score_i = -PH_BLEND_BASE_GAP * original_rank_i
    """
    if candidates is None or len(candidates) <= 1:
        dbg = [{"rank": 1, "old_idx": 0, "score": 0.0}] if candidates else []
        return candidates, dbg

    n = len(candidates)

    # If GUDHI is unavailable, keep the original order.
    if not GUDHI_AVAILABLE:
        dbg = [{
            "rank": i + 1,
            "old_idx": i,
            "base_score": -PH_BLEND_BASE_GAP * i,
            "ph_raw_score": 0.0,
            "ph_norm_score": 0.0,
            "score": -PH_BLEND_BASE_GAP * i,
        } for i in range(n)]
        return candidates, dbg

    ref_feat = _make_reference_ph_feature(candidates)

    raw_rows = []
    for i, c in enumerate(candidates):
        try:
            ph_raw, aux = ph_score_candidate(c, ref_feat)
        except Exception as e:
            ph_raw = -1e18
            aux = {
                "ph_dist": np.inf,
                "bond_pen": np.inf,
                "clash_pen": np.inf,
                "score": ph_raw,
                "error": str(e),
            }
        raw_rows.append((i, c, ph_raw, aux))

    ph_vals = np.array([r[2] for r in raw_rows], dtype=np.float32)
    finite_mask = np.isfinite(ph_vals) & (ph_vals > -1e17)

    ph_norm = np.zeros(n, dtype=np.float32)
    if finite_mask.sum() >= 2:
        mu = float(ph_vals[finite_mask].mean())
        sd = float(ph_vals[finite_mask].std())
        if sd > 1e-8:
            ph_norm[finite_mask] = (ph_vals[finite_mask] - mu) / sd
    elif finite_mask.sum() == 1:
        ph_norm[finite_mask] = 0.0

    scored = []
    for i, c, ph_raw, aux in raw_rows:
        base_score = -PH_BLEND_BASE_GAP * float(i)
        final_score = base_score + PH_BLEND_LAMBDA * float(ph_norm[i])
        row_aux = {
            **aux,
            "base_score": base_score,
            "ph_raw_score": float(ph_raw) if np.isfinite(ph_raw) else ph_raw,
            "ph_norm_score": float(ph_norm[i]),
            "score": final_score,
        }
        scored.append((final_score, i, c, row_aux))

    scored.sort(key=lambda z: z[0], reverse=True)
    reranked = [z[2] for z in scored]

    dbg = []
    for rank, (final_score, old_idx, _, aux) in enumerate(scored, start=1):
        dbg.append({"rank": rank, "old_idx": old_idx, **aux})

    if verbose:
        tag = f"[PH blend rerank][{target_id}]" if target_id is not None else "[PH blend rerank]"
        print(tag)
        for d in dbg[:min(5, len(dbg))]:
            print(
                f"  rank={d['rank']} old_idx={d['old_idx']} "
                f"final={d['score']:.4f} "
                f"base={d['base_score']:.4f} "
                f"ph_norm={d['ph_norm_score']:.4f} "
                f"ph_raw={d['ph_raw_score']:.4f}"
            )

    # ── パーシステント図を描画（全候補） ─────────────────────────────────
    if GUDHI_AVAILABLE:
        tid_label = target_id.replace("::", "_") if target_id else "unknown"
        for rank_info in dbg:
            s_idx   = rank_info["old_idx"]
            c_orig  = candidates[s_idx] if s_idx < len(candidates) else reranked[0]
            h1p     = _get_h1_pairs_for_coords(c_orig)
            plot_persistence_diagram(h1p, tid_label, s_idx)

    return reranked, dbg


# ─────────────── Main ────────────────────────────────────────────────────────
def main() -> None:
    test_csv, output_csv, code_dir, root_dir = resolve_paths()

    if not os.path.isdir(code_dir):
        raise FileNotFoundError(
            f"Missing PROTENIX_CODE_DIR: {code_dir}. "
            "Set PROTENIX_CODE_DIR to the repo path."
        )

    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    sys.path.append(code_dir)
    ensure_required_files(root_dir)
    seed_everything(SEED)

    # ── Load test data ──────────────────────────────────────────────────────
    test_df_full = pd.read_csv(test_csv)
    test_df      = (test_df_full.head(LOCAL_N_SAMPLES) if not IS_KAGGLE
                    else test_df_full).reset_index(drop=True)
    print(f"Test targets : {len(test_df)}"
          + (" (LOCAL MODE)" if not IS_KAGGLE else ""))

    seq_by_id = dict(zip(test_df["target_id"], test_df["sequence"]))

    # Truncated copy for Protenix (Protenix has token limits)
    test_df_trunc = test_df.copy()
    test_df_trunc["sequence"] = test_df_trunc["sequence"].str[:MAX_SEQ_LEN]

    # ── Load training data for TBM ──────────────────────────────────────────
    print("\nLoading training data for TBM …")
    train_seqs   = pd.read_csv(DEFAULT_TRAIN_CSV)
    val_seqs     = pd.read_csv(DEFAULT_VAL_CSV)
    train_labels = pd.read_csv(DEFAULT_TRAIN_LBLS)
    val_labels   = pd.read_csv(DEFAULT_VAL_LBLS)

    combined_seqs   = pd.concat([train_seqs,   val_seqs],    ignore_index=True)
    combined_labels = pd.concat([train_labels, val_labels],  ignore_index=True)
    train_coords    = process_labels(combined_labels)
    segments_map, _ = build_segments_map(test_df)

    print(f"Template pool: {len(combined_seqs)} sequences, {len(train_coords)} structures")

    # ─── PHASE 1: TBM ──────────────────────────────────────────────────────
    template_preds, protenix_queue = tbm_phase(
        test_df, combined_seqs, train_coords, segments_map
    )

    # ─── PHASE 2: Protenix (only for targets that need extra predictions) ──
    protenix_preds: dict = {}   # target_id -> np.ndarray (n_needed, seq_len, 3)

    if protenix_queue and USE_PROTENIX:
        print(f"\n{'='*60}")
        print(f"PHASE 2: Protenix for {len(protenix_queue)} targets")
        print(f"{'='*60}")

        work_dir = Path("/kaggle/working")
        work_dir.mkdir(parents=True, exist_ok=True)

        # ── 1. Preparation: create tasks for all sequences/chunks ────────────
        tasks = []          # list of dict for input_json
        chunk_info = {}     # target_id -> list of {"name": chunk_name, "range": (s, e)}
        
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq)
            if seq_len <= MAX_SEQ_LEN:
                tasks.append({"target_id": target_id, "sequence": full_seq})
                chunk_info[target_id] = [{"name": target_id, "range": (0, seq_len)}]
                print(f"  {target_id} ({seq_len} nt): single pass queued")
            else:
                chunks = split_into_chunks(seq_len, MAX_SEQ_LEN, CHUNK_OVERLAP)
                print(f"  {target_id} ({seq_len} nt): {len(chunks)} chunks queued "
                      f"{[(s, e) for s, e in chunks]}")
                
                chunk_info[target_id] = []
                for ci, (cs, ce) in enumerate(chunks):
                    chunk_name = f"{target_id}_chunk{ci}"
                    sub_seq = full_seq[cs:ce]
                    tasks.append({"target_id": chunk_name, "sequence": sub_seq})
                    chunk_info[target_id].append({"name": chunk_name, "range": (cs, ce)})

        # Build combined input JSON
        tasks_df = pd.DataFrame(tasks)
        input_json_path = str(work_dir / "protenix_queue_input.json")
        build_input_json(tasks_df, input_json_path)

        from protenix.data.inference.infer_dataloader import InferenceDataset
        from runner.inference import (InferenceRunner,
                                      update_gpu_compatible_configs,
                                      update_inference_configs)

        # Initialize model exactly ONCE
        configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
        configs = update_gpu_compatible_configs(configs)
        runner  = InferenceRunner(configs)
        dataset = InferenceDataset(configs)

        # ── 2. Inference: process dataset and collect predictions ────────────
        raw_predictions = {}  # sample_name -> coords (np.ndarray or None)

        def _extract_c1_coords(prediction, feat, chunk_seq_len, raw_coords):
            if "centre_atom_mask" in feat:
                mask = (feat["centre_atom_mask"] == 1).to(raw_coords.device)
            elif "atom_to_tokatom_idx" in feat:
                m11 = (feat["atom_to_tokatom_idx"] == 11).to(raw_coords.device)
                m12 = (feat["atom_to_tokatom_idx"] == 12).to(raw_coords.device)
                c11, c12 = m11.sum(), m12.sum()
                mask = m11 if abs(c11 - chunk_seq_len) < abs(c12 - chunk_seq_len) else m12
            else:
                mask = torch.zeros(raw_coords.shape[1], dtype=torch.bool, device=raw_coords.device)
            
            coords = raw_coords[:, mask, :].detach().cpu().numpy()
            
            # Collapse check
            if coords.shape[1] > 1:
                diffs = np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1)
                if np.all(diffs < 1e-4):
                    print(f"    WARNING: Collapsed coordinates detected")
                    return None
            
            if coords.shape[1] != chunk_seq_len:
                if coords.shape[1] == 1 and chunk_seq_len > 1:
                    return None
                padded = np.zeros((coords.shape[0], chunk_seq_len, 3), dtype=np.float32)
                ml = min(coords.shape[1], chunk_seq_len)
                padded[:, :ml, :] = coords[:, :ml, :]
                coords = padded
            return coords

        for i in tqdm(range(len(dataset)), desc="Protenix Inference"):
            data, atom_array, err = dataset[i]
            sample_name = data.get("sample_name", f"sample_{i}")
            
            if err:
                print(f"  {sample_name} data error: {err}")
                raw_predictions[sample_name] = None
                del data, atom_array, err
                gc.collect(); torch.cuda.empty_cache(); gc.collect()
                continue
            
            # Find how many samples are needed for this specific query
            target_id = sample_name.split("_chunk")[0] if "_chunk" in sample_name else sample_name
            n_needed = protenix_queue.get(target_id, (N_SAMPLE, ""))[0]
            sub_seq_len = data["N_token"].item() # roughly correct
            
            try:
                new_cfg = update_inference_configs(configs, sub_seq_len)
                new_cfg.sample_diffusion.N_sample = n_needed
                runner.update_model_configs(new_cfg)
                
                pred = runner.predict(data)
                raw_coords = pred["coordinate"]
                
                coords = _extract_c1_coords(pred, data["input_feature_dict"], 
                                            sub_seq_len, raw_coords)

                # ── pLDDT抽出してサンプルを降順ソート ──────────────────────
                if coords is not None:
                    plddt_raw = None
                    for key in ("plddt", "atom_plddt", "confidence_score",
                                "predicted_lddt", "plddts"):
                        if key in pred:
                            plddt_raw = pred[key]
                            print(f"[DEBUG] pLDDT key='{key}' shape={plddt_raw.shape}")
                            break
                    if plddt_raw is None:
                        print("[DEBUG] pLDDT not found:", list(pred.keys()))

                    plddt_scores = None
                    if plddt_raw is not None:
                        try:
                            p = plddt_raw.detach().cpu().numpy()
                            n_s = coords.shape[0]
                            if p.ndim == 2 and p.shape[0] == n_s:
                                plddt_scores = p.mean(axis=-1)
                            elif p.ndim == 2 and p.shape[1] == n_s:
                                plddt_scores = p.T.mean(axis=-1)
                            elif p.ndim == 1 and len(p) == n_s:
                                plddt_scores = p
                            else:
                                print(f"[DEBUG] pLDDT shape {p.shape} incompatible")
                        except Exception as e:
                            print(f"[DEBUG] pLDDT failed: {e}")

                    if plddt_scores is not None and len(plddt_scores) == coords.shape[0]:
                        order  = np.argsort(plddt_scores)[::-1]
                        coords = coords[order]
                        print(f"  {sample_name}: sorted by pLDDT "
                              f"{plddt_scores[order].round(1).tolist()}")

                raw_predictions[sample_name] = coords
            except Exception as exc:
                print(f"  {sample_name} inference failed: {exc}")
                import traceback; traceback.print_exc()
                raw_predictions[sample_name] = None
            finally:
                try: del pred, data, atom_array, raw_coords
                except: pass
                gc.collect(); torch.cuda.empty_cache(); gc.collect()

        # ── 3. Post-processing: Stitching and final formatting ───────────────
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq)
            chunks = chunk_info.get(target_id, [])
            
            if not chunks:
                continue

            if len(chunks) == 1:
                # Single pass
                coords = raw_predictions.get(target_id)
                protenix_preds[target_id] = coords
                if coords is not None:
                    print(f"  {target_id}: {coords.shape[0]} predictions generated")
                else:
                    print(f"  {target_id}: FAILED")
            else:
                # Stitch chunks together
                chunk_results_per_sample = {s: [] for s in range(n_needed)}
                all_ok = True
                
                for ci, cinfo in enumerate(chunks):
                    cname = cinfo["name"]
                    crange = cinfo["range"]
                    ccoords = raw_predictions.get(cname)
                    
                    if ccoords is None:
                        all_ok = False
                        break
                    
                    for s_idx in range(n_needed):
                        if s_idx < ccoords.shape[0]:
                            chunk_results_per_sample[s_idx].append((ccoords[s_idx], crange))
                        else:
                            chunk_results_per_sample[s_idx].append((ccoords[-1], crange))
                
                if not all_ok:
                    print(f"  {target_id}: chunked inference incomplete, using fallback")
                    protenix_preds[target_id] = None
                    continue
                
                stitched_samples = []
                for s_idx in range(n_needed):
                    items = chunk_results_per_sample[s_idx]
                    coords_list = [c for c, _ in items]
                    ranges_list = [r for _, r in items]
                    full_coords = stitch_chunk_coords(coords_list, ranges_list, seq_len)
                    stitched_samples.append(full_coords)
                
                result = np.stack(stitched_samples, axis=0)
                protenix_preds[target_id] = result
                print(f"  {target_id}: {result.shape[0]} stitched predictions generated")
# ...existing code...

    elif protenix_queue and not USE_PROTENIX:
        print(f"\nPHASE 2 skipped (USE_PROTENIX=False). "
              f"De-novo fallback will cover {len(protenix_queue)} targets.")

    # ─── PHASE 3: Combine everything ───────────────────────────────────────
    print(f"\n{'='*60}")
    print("PHASE 3: Combine TBM + Protenix + de-novo fallback")
    print(f"{'='*60}")

    all_rows = []

    for _, row in test_df.iterrows():
        tid = row["target_id"]
        seq = row["sequence"]

        combined: list = list(template_preds.get(tid, []))  # TBM predictions (keep original order)

        # Append Protenix predictions to fill remaining slots.
        # PH is applied ONLY inside the Protenix candidate block, so TBM ordering is preserved.
        ptx = protenix_preds.get(tid)
        if ptx is not None and ptx.ndim == 3:
            ptx_list = [ptx[j] for j in range(ptx.shape[0])]

            if PH_RERANK_ENABLED and len(ptx_list) > 1:
                ptx_list, ph_dbg = ph_rerank_candidates(
                    ptx_list,
                    target_id=f"{tid}::Protenix",
                    verbose=True,   # Always draw diagram
                )

            for cand in ptx_list:
                if len(combined) >= N_SAMPLE:
                    break
                combined.append(cand)  # (seq_len, 3)

        # De-novo fallback for any still-empty slots
        n_denovo = 0
        while len(combined) < N_SAMPLE:
            seed_val = row.name * 1000000 + len(combined) * 1000
            dn       = generate_rna_structure(seq, seed=seed_val)
            combined.append(adaptive_rna_constraints(dn, tid, segments_map, confidence=0.2))
            n_denovo += 1

        if n_denovo:
            print(f"  {tid}: {n_denovo} slot(s) filled with de-novo fallback")

        # Stack to (N_SAMPLE, seq_len, 3) and write rows
        stacked = np.stack(combined[:N_SAMPLE], axis=0)
        all_rows.extend(coords_to_rows(tid, seq, stacked))

    # ── Save ───────────────────────────────────────────────────────────────
    sub = pd.DataFrame(all_rows)
    cols = ["ID", "resname", "resid"] + [
        f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]
    ]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
    sub[cols].to_csv(output_csv, index=False)

    print(f"\n✓ Saved submission to {output_csv}  ({len(sub):,} rows)")


# Main

In [ ]:

if __name__ == "__main__":
    main()

In [ ]:
#read submission.csv
submission_path = "/kaggle/working/submission.csv"
submission_df = pd.read_csv(submission_path)
print(submission_df.head(20))

In [ ]:
'''# ============================================================
# Post-submission noise augmentation (reusable)
# ============================================================
# The main pipeline outputs submission_org.csv.
# This cell adds noise to generate submission.csv (for final submission).
# The noise is saved to noise_record.npz and can be reused on subsequent runs.
# ============================================================

import numpy as np
import pandas as pd
import os

NOISE_SEED     = 42          # ← Change this for a different noise pattern
NOISE_STD      = 0.1         # Standard deviation of added noise (Å)
SUBMISSION_IN  = "/kaggle/working/submission.csv"   # メイン出力（ノイズなし）
SUBMISSION_OUT = "/kaggle/working/submission.csv"        # 最終提出用（ノイズあり）
NOISE_SAVE     = "/kaggle/working/noise_record.npz"
NOISE_LOAD     = "/kaggle/input/datasets/takahiro3110/v32-noise/noise_record.npz" 

sub = pd.read_csv(SUBMISSION_IN)

coord_cols = [c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]
values = sub[coord_cols].values.astype(np.float64)

if NOISE_LOAD and os.path.exists(NOISE_LOAD):
    loaded = np.load(NOISE_LOAD)
    noise  = loaded["noise"]
    assert noise.shape == values.shape, (
        f"Noise shape mismatch: saved={noise.shape}, current={values.shape}"
    )
    print(f"✓ Noise reused: {NOISE_LOAD}  shape={noise.shape}")
else:
    rng   = np.random.default_rng(NOISE_SEED)
    noise = rng.normal(0.0, NOISE_STD, values.shape)
    np.savez_compressed(
        NOISE_SAVE,
        noise      = noise,
        coord_cols = np.array(coord_cols),
        seed       = np.array([NOISE_SEED]),
        std        = np.array([NOISE_STD]),
    )
    print(f"✓ Noise generated and saved: {NOISE_SAVE}  shape={noise.shape}  std={NOISE_STD}")

sub_out = sub.copy()
sub_out[coord_cols] = values + noise
sub_out.to_csv(SUBMISSION_OUT, index=False)

print(f"✓ Saved: {SUBMISSION_OUT}")
print(f"  Rows: {len(sub_out)}")
print(f"  Noise stats: mean={noise.mean():.4f}  std={noise.std():.4f}  max_abs={np.abs(noise).max():.4f}")
print()
print("How to reuse:")
print("  Change NOISE_LOAD = \"/kaggle/working/noise_record.npz\" and re-run")'''

In [ ]:
# ============================================================
# Display the 5 most recently created persistent diagrams
# ============================================================
import glob, os
from IPython.display import display, Image

png_files = sorted(
    glob.glob("/kaggle/working/ph_diagram_*.png"),
    key=os.path.getmtime,
    reverse=True
)[:5]

if png_files:
    print(f"Displaying {len(png_files)} most recent file(s) (newest first)")
    for path in png_files:
        print(f"  {os.path.basename(path)}")
        display(Image(filename=path))
else:
    print("No ph_diagram_*.png files found.")
    print("Please ensure PH reranking is enabled (GUDHI_AVAILABLE=True) and has been run.")